<a href="https://colab.research.google.com/github/OlajideFemi/Carbon-Footprint/blob/main/Mojibake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [ ]:
df.to_csv("salesforce_clean.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd
import unicodedata

def clean_text(value):
    if isinstance(value, str):
        value = unicodedata.normalize("NFKD", value)
        value = value.replace("Â", "")
        value = value.replace("Ã", "")
        value = value.replace("â€™", "'")
        value = value.replace("â€“", "-")
        value = value.strip()
    return value

df = df.applymap(clean_text)

In [ ]:
import re

def remove_weird_chars(text):
    if isinstance(text, str):
        return re.sub(r"[^\x20-\x7E]+", "", text)
    return text

df = df.applymap(remove_weird_chars)

In [ ]:
from simple_salesforce import Salesforce
import pandas as pd

data = sf.query_all("SELECT Id, Name FROM Account")

df = pd.DataFrame(data["records"]).drop(columns=["attributes"])

df.to_csv("salesforce_data.csv", index=False, encoding="utf-8-sig")

BrokenActualâ€™’â€“–Ã©éÂ££

In [ ]:
df = df.applymap(lambda x: x.encode('latin1').decode('utf8') if isinstance(x,str) else x)

In [ ]:
def clean_salesforce_dataframe(df):

    import unicodedata
    import re

    def clean(value):
        if isinstance(value, str):
            value = unicodedata.normalize("NFKC", value)
            value = re.sub(r"[^\x20-\x7E£€]", "", value)
            value = value.strip()
        return value

    return df.applymap(clean)

df = clean_salesforce_dataframe(df)

In [ ]:
import pandas as pd
import re
import unicodedata


def fix_mojibake(text):
    """
    Try to repair text that was decoded with the wrong encoding.
    Example: 'â€™' -> '’', 'Ã©' -> 'é', 'Â£' -> '£'
    """
    if not isinstance(text, str):
        return text

    original = text

    # Only attempt repair if common mojibake markers appear
    suspicious_markers = ["Ã", "Â", "â", "ð", "�"]
    if any(marker in text for marker in suspicious_markers):
        for src_enc, dst_enc in [("latin1", "utf-8"), ("cp1252", "utf-8")]:
            try:
                repaired = text.encode(src_enc, errors="ignore").decode(dst_enc, errors="ignore")
                if repaired and repaired != text:
                    text = repaired
                    break
            except Exception:
                pass

    return text if text else original


def clean_unicode_text(text):
    """
    General text cleanup after mojibake repair.
    """
    if not isinstance(text, str):
        return text

    # Repair bad decoding first
    text = fix_mojibake(text)

    # Normalize unicode
    text = unicodedata.normalize("NFKC", text)

    # Replace non-breaking spaces with normal spaces
    text = text.replace("\u00A0", " ")

    # Remove zero-width / invisible characters
    text = re.sub(r"[\u200B-\u200D\uFEFF]", "", text)

    # Remove ASCII control chars except tab/newline if you want to keep them
    text = re.sub(r"[\x00-\x1F\x7F]", "", text)

    # Strip outer whitespace
    text = text.strip()

    return text


def clean_salesforce_dataframe(df):
    """
    Clean all object/string columns in a DataFrame.
    """
    df = df.copy()

    object_cols = df.select_dtypes(include=["object", "string"]).columns

    for col in object_cols:
        df[col] = df[col].map(clean_unicode_text)

    return df


# Example usage
# df = pd.DataFrame(sf.query_all(query)["records"]).drop(columns=["attributes"], errors="ignore")
# df = clean_salesforce_dataframe(df)
# df.to_csv("salesforce_clean.csv", index=False, encoding="utf-8-sig")

In [ ]:
def clean_salesforce_dataframe_with_report(df, sample_rows=5):
    cleaned_df = clean_salesforce_dataframe(df)

    object_cols = df.select_dtypes(include=["object", "string"]).columns
    changes = []

    for col in object_cols:
        before = df[col]
        after = cleaned_df[col]

        changed_mask = before.fillna("") != after.fillna("")
        changed_rows = df.loc[changed_mask, [col]].head(sample_rows)

        if not changed_rows.empty:
            for idx in changed_rows.index:
                changes.append({
                    "row_index": idx,
                    "column": col,
                    "before": before.loc[idx],
                    "after": after.loc[idx],
                })

    report_df = pd.DataFrame(changes)
    return cleaned_df, report_df


# Example:
# cleaned_df, report = clean_salesforce_dataframe_with_report(df)
# print(report.head(20))
# cleaned_df.to_csv("salesforce_clean.csv", index=False, encoding="utf-8-sig")

In [ ]:
from simple_salesforce import Salesforce
import pandas as pd

# Example query
result = sf.query_all("""
    SELECT Id, Name, BillingCity, Description
    FROM Account
""")

df = pd.DataFrame(result["records"]).drop(columns=["attributes"], errors="ignore")

# Clean text problems
df = clean_salesforce_dataframe(df)

# Export for Excel
df.to_csv("salesforce_clean.csv", index=False, encoding="utf-8-sig")

In [ ]:
print(df.head().to_dict())

In [2]:
import pandas as pd
import re
import unicodedata


MOJIBAKE_HINTS = ("Ã", "Â", "â", "ð", "�")


def looks_broken(text: str) -> bool:
    if not isinstance(text, str) or not text:
        return False
    return any(h in text for h in MOJIBAKE_HINTS)


def safe_roundtrip_repair(text: str) -> str:
    """
    Try common wrong-decoding reversals.
    """
    candidates = [text]

    transforms = [
        ("latin1", "utf-8"),
        ("cp1252", "utf-8"),
        ("latin1", "cp1252"),
    ]

    for src, dst in transforms:
        try:
            repaired = text.encode(src, errors="ignore").decode(dst, errors="ignore")
            if repaired:
                candidates.append(repaired)
        except Exception:
            pass

    # Try a second pass for double-broken text
    for candidate in list(candidates):
        for src, dst in transforms:
            try:
                repaired = candidate.encode(src, errors="ignore").decode(dst, errors="ignore")
                if repaired:
                    candidates.append(repaired)
            except Exception:
                pass

    return pick_best_candidate(candidates)


def pick_best_candidate(candidates):
    """
    Pick the least broken-looking candidate.
    """
    def score(s):
        if not isinstance(s, str):
            return 10**9

        bad_markers = sum(s.count(x) for x in ["Ã", "Â", "â", "ð", "�"])
        control_chars = len(re.findall(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]", s))
        weird_ratio = sum(1 for ch in s if ord(ch) < 32 and ch not in "\t\n\r")
        return bad_markers * 10 + control_chars * 5 + weird_ratio

    return min(candidates, key=score)


def normalize_common_symbols(text: str) -> str:
    replacements = {
        "\u2018": "'",   # left single quote
        "\u2019": "'",   # right single quote
        "\u201C": '"',   # left double quote
        "\u201D": '"',   # right double quote
        "\u2013": "-",   # en dash
        "\u2014": "-",   # em dash
        "\u2026": "...", # ellipsis
        "\u00A0": " ",   # non-breaking space
        "\u200B": "",    # zero-width space
        "\u200C": "",
        "\u200D": "",
        "\uFEFF": "",    # BOM
        "\x00": "",      # null byte
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def strip_control_chars(text: str) -> str:
    return re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]", "", text)


def salvage_text(text):
    if pd.isna(text):
        return text

    if not isinstance(text, str):
        return text

    original = text

    # Step 1: remove obvious invisible junk
    text = normalize_common_symbols(text)
    text = strip_control_chars(text)

    # Step 2: normalize unicode form
    text = unicodedata.normalize("NFKC", text)

    # Step 3: attempt encoding repair only when suspicious
    if looks_broken(text):
        text = safe_roundtrip_repair(text)

    # Step 4: normalize again after repair
    text = unicodedata.normalize("NFKC", text)
    text = normalize_common_symbols(text)
    text = strip_control_chars(text)

    # Step 5: collapse whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = text.strip()

    return text if text != "" else original


def clean_salesforce_df_extreme(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    obj_cols = df.select_dtypes(include=["object", "string"]).columns

    for col in obj_cols:
        df[col] = df[col].map(salvage_text)

    return df


def audit_repairs(before_df: pd.DataFrame, after_df: pd.DataFrame, max_rows=100):
    changes = []

    common_cols = [c for c in before_df.columns if c in after_df.columns]

    for col in common_cols:
        before = before_df[col]
        after = after_df[col]

        if str(before.dtype) not in ("object", "string") and str(after.dtype) not in ("object", "string"):
            continue

        mask = before.fillna("").astype(str) != after.fillna("").astype(str)

        for idx in before_df.index[mask][:max_rows]:
            changes.append({
                "row_index": idx,
                "column": col,
                "before": before.loc[idx],
                "after": after.loc[idx],
            })

    return pd.DataFrame(changes)

In [ ]:
# Build DataFrame from Salesforce
df = pd.DataFrame(result["records"]).drop(columns=["attributes"], errors="ignore")

# Keep original for comparison
raw_df = df.copy()

# Clean
df = clean_salesforce_df_extreme(df)

# Audit what changed
repair_report = audit_repairs(raw_df, df)
print(repair_report.head(20))

# Export for Excel
df.to_csv("salesforce_clean_extreme.csv", index=False, encoding="utf-8-sig")
repair_report.to_csv("salesforce_repair_report.csv", index=False, encoding="utf-8-sig")

In [ ]:
def flag_suspicious_cells(df):
    flagged = []

    obj_cols = df.select_dtypes(include=["object", "string"]).columns

    pattern = r"(Ã|Â|â|ð|�|\?{2,})"

    for col in obj_cols:
        mask = df[col].astype(str).str.contains(pattern, regex=True, na=False)
        for idx in df.index[mask]:
            flagged.append({
                "row_index": idx,
                "column": col,
                "value": df.at[idx, col],
            })

    return pd.DataFrame(flagged)

suspicious_report = flag_suspicious_cells(df)
suspicious_report.to_csv("salesforce_suspicious_rows.csv", index=False, encoding="utf-8-sig")